# Chapter 2: carry the method spread into RDF
Kestrel Lager: OG 12.0, AE 2.2 degrees Plato. Repeatability sd is 0.05 on AE and 0.015 on OG. How far does RDF move by chance? The notebook also draws Figure 2.3 and the Poisson intervals from 2.4.2.

**No code tier.** In a spreadsheet, put OG in A1 and AE in B1. Work RE, ABW and RDF in three cells with the MBAA relations. Change B1 to 2.25 and read the new RDF.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"
from scipy import stats

In [ ]:
def rdf(og, ae):
    re = 0.1808 * og + 0.8192 * ae
    abw = (og - re) / (2.0665 - 0.010665 * og)
    return 100 * 2.0665 / (2.0665 + re / abw)

print("RDF", round(rdf(12.0, 2.2), 1), "| apparent attenuation", round(100 * (1 - 2.2 / 12.0), 1))
print("AE 2.25:", round(rdf(12.0, 2.25), 1), "| OG + 0.015:", round(rdf(12.015, 2.2) - rdf(12.0, 2.2), 3))

In [ ]:
rng = np.random.default_rng(2)
sim = rdf(rng.normal(12.0, 0.015, 200_000), rng.normal(2.2, 0.05, 200_000))
print("RDF sd by simulation:", round(sim.std(), 2), "| two results may differ by", round(2.8 * sim.std(), 1), "points")
plt.hist(sim, bins=80, color="#2f7f72"); plt.xlabel("RDF (percent)"); plt.title("RDF from one beer, measured many times (synthetic)"); plt.show()

In [ ]:
result, limit = 38, 50
fig, ax = plt.subplots(figsize=(7, 1.8))
for sd, colour, label in ((6, "#cfe6e1", "between labs sd 6"), (3, "#2f7f72", "repeatability sd 3")):
    ax.barh(0, 2 * 1.96 * sd, left=result - 1.96 * sd, color=colour, label=label)
ax.axvline(result, color="#0b4f45", lw=2); ax.axvline(limit, color="#b23a2e", ls="--", label="limit 50 ppb")
ax.set_yticks([]); ax.set_xlabel("Dissolved oxygen (ppb)"); ax.legend(loc="upper left", fontsize=8); plt.show()

In [ ]:
for count in (1, 4):
    lo, hi = stats.chi2.ppf(0.025, 2 * count) / 2, stats.chi2.ppf(0.975, 2 * count + 2) / 2
    print(f"count {count}: exact 95 percent interval {lo:.2f} to {hi:.1f}")
print("clean plate at a true 1 CFU:", round(100 * math.exp(-1)), "percent")

**What to look at.** The simulated sd sits near 0.33 points, so an RDF change under a point is not a finding. The count of 4 sits under the action limit of 5, yet its interval reaches past it.